#  Rossmann Store Sales Prediction

##  What is this project about?

Imagine you manage **1,115 drug stores across Germany** (the Rossmann chain). Every day, each store has a different number of sales depending on:
-  The day of the week
-  Holidays
-  Whether there's a promotion
-  The type of store
-  How close the competition is

 **The challenge:** *How many sales will each store make tomorrow, next week, or next month?*

 **Why this matters:** Knowing future sales helps the company:
-  Order the right amount of inventory (no stockouts, no waste)
-  Schedule the right number of staff
-  Plan budgets accurately
-  Decide which stores need promotions

---

##  What this notebook does (step by step)

1.  **Load** sales and store data (1+ million rows!)
2.  **Merge** the two datasets together
3.  **Clean** the data — handle missing values
4.  **Engineer features** — create useful columns from existing ones
5.  **Visualize** sales patterns
6.  **Add lag features** — what were sales 7, 14, 30 days ago?
7.  **Time-based train/test split** — train on past, test on future
8.  **Train 2 models** — Linear Regression and XGBoost
9.  **Evaluate** with MAE, RMSE, R²
10.  **Find what drives sales** (feature importance)
11.  **Save** the model and predictions

---

##  Who is this notebook for?

**Anyone curious about real-world ML!** Even if you've never coded before, every step explains:
-  **What** the code is doing
-  **Why** we're doing it
-  **What the output means** in plain English

## Step 0: Install all libraries (one line!)

Before we can do anything, we need to install the **"tools"** (libraries) Python will use.

Think of these like apps on your phone — each one does a specific job:

| Library | What it does (in simple terms) |
|---------|-------------------------------|
| **pandas** | Like Excel, but for Python — handles data tables |
| **numpy** | A super-fast calculator for numbers |
| **matplotlib** | Draws charts and graphs |
| **seaborn** | Makes those charts look prettier |
| **scikit-learn** | A big toolbox of ML algorithms |
| **xgboost** | A powerful prediction algorithm (used by Kaggle winners ) |
| **joblib** | Saves trained models to disk |

 **You only need to run this once** — Python remembers the libraries afterward.

In [ ]:
!pip install pandas numpy matplotlib seaborn scikit-learn xgboost joblib

## Step 1: Import the libraries

Now we tell Python: *"I want to use these tools in this notebook."*

It's like opening apps on your phone before using them.

 **About `warnings.filterwarnings("ignore")`:** Python sometimes shows yellow warning messages (just suggestions, not errors). We hide them to keep our notebook tidy.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor
import warnings
warnings.filterwarnings("ignore")

## Step 2: Load the data

We have **two CSV files** to load:

###  `train.csv` — Daily sales data
Each row = one store on one day. Includes:
- Store ID
- Date
- Sales (the number we want to predict!)
- Customers, Promotions, Holidays, etc.

###  `store.csv` — Store info
Each row = one store. Includes:
- Store type (a, b, c, d)
- Assortment level
- Distance to nearest competitor
- Promotion details

 **About `parse_dates=["Date"]`:** Tells pandas *"the Date column is actually dates, not text."* This unlocks date-related features later (like extracting year, month, day-of-week).

 **About the path:** `r"..."` is a *raw string* — needed for Windows paths. Update it to wherever you saved your files.

In [ ]:
train = pd.read_csv(r"C:\Users\sg961\Downloads\archive\rossmann-store-sales\train.csv", parse_dates=["Date"], low_memory=False)
store = pd.read_csv(r"C:\Users\sg961\Downloads\archive\rossmann-store-sales\store.csv")

print("Train shape:", train.shape)
print("Store shape:", store.shape)
train.head()

## Step 3: Merge the two datasets

Right now we have:
- **Train data** with daily sales (but no info about each store)
- **Store data** with store details (but no daily sales)

We need to **combine them** into one big table where each row has BOTH the day's sales AND the store info.

 **Think of it like VLOOKUP in Excel:** for every row in `train`, find the matching `Store` ID in `store`, and copy over those columns.

###  We also check:
- **Total rows after merging** (should be ~1 million)
- **Data types** of each column
- **Missing values** — empty cells we'll need to fix

In [ ]:
df = train.merge(store, on="Store", how="left")
print(df.shape)
print(df.info())
print(df.isnull().sum())
df.head()

## Step 4: Clean the data

Real-world data is **messy**. We need to fix several issues:

###  Remove useless rows
**Closed stores have $0 sales** — that's not useful for our model. We only keep rows where:
- The store was **open** (`Open == 1`)
- It actually had **some sales** (`Sales > 0`)

###  Fill missing values
Some columns have empty cells. We replace them with sensible defaults:

| Column | Why it might be empty | What we fill with |
|--------|----------------------|-------------------|
| `CompetitionDistance` | No competition info | The **median** distance |
| `CompetitionOpenSinceMonth/Year` | No competitor nearby | `0` |
| `Promo2SinceWeek/Year` | Store doesn't run Promo2 | `0` |
| `PromoInterval` | No recurring promos | `"None"` |

###  Sort by store and date
Crucial for time-series features later — we need rows in order!

In [ ]:
# Keep only days the store was open and had sales
df = df[(df["Open"] == 1) & (df["Sales"] > 0)]

# Fill missing competition/promo info
df["CompetitionDistance"] = df["CompetitionDistance"].fillna(df["CompetitionDistance"].median())
df["CompetitionOpenSinceMonth"] = df["CompetitionOpenSinceMonth"].fillna(0)
df["CompetitionOpenSinceYear"] = df["CompetitionOpenSinceYear"].fillna(0)
df["Promo2SinceWeek"] = df["Promo2SinceWeek"].fillna(0)
df["Promo2SinceYear"] = df["Promo2SinceYear"].fillna(0)
df["PromoInterval"] = df["PromoInterval"].fillna("None")

df = df.sort_values(["Store", "Date"]).reset_index(drop=True)
print("After cleaning:", df.shape)

## Step 5: Feature engineering — extract date info

 **Big idea:** Right now, the `Date` column is just "2015-07-31". But the model can learn way more if we **break it apart** into useful pieces.

###  Date features we create:

| Feature | What it captures |
|---------|------------------|
| `Year` | Long-term trends (sales growing year over year?) |
| `Month` | Seasonal patterns (Christmas? Summer slump?) |
| `Day` | Beginning vs end of month |
| `WeekOfYear` | Specific weekly patterns |
| `DayOfWeek` | Weekdays vs weekends |
| `IsWeekend` | Quick weekend flag (1 = Sat/Sun, 0 = weekday) |

###  Convert text to numbers
Models can't read text like "a", "b", "c". We **map them to numbers**:
- `StateHoliday`: `"0"→0, "a"→1, "b"→2, "c"→3`
- `StoreType`: `a→0, b→1, c→2, d→3`
- `Assortment`: `a→0, b→1, c→2`

 **Why this is OK here:** these categories don't have meaningful order, but XGBoost (a tree-based model) handles encoded categories just fine.

In [ ]:
# Date features
df["Year"] = df["Date"].dt.year
df["Month"] = df["Date"].dt.month
df["Day"] = df["Date"].dt.day
df["WeekOfYear"] = df["Date"].dt.isocalendar().week.astype(int)
df["DayOfWeek"] = df["Date"].dt.dayofweek
df["IsWeekend"] = (df["DayOfWeek"] >= 5).astype(int)

# Encode categorical columns
df["StateHoliday"] = df["StateHoliday"].astype(str).map({"0": 0, "a": 1, "b": 2, "c": 3}).fillna(0)
df["StoreType"] = df["StoreType"].map({"a": 0, "b": 1, "c": 2, "d": 3})
df["Assortment"] = df["Assortment"].map({"a": 0, "b": 1, "c": 2})

print(df[["Date", "Sales", "Store", "DayOfWeek", "Promo", "StoreType"]].head())

## Step 6: Visualize — see patterns in the sales

Charts help us **see patterns** that numbers alone hide.

###  Chart 1: Average daily sales over time
Shows the overall trend — is the company growing? Are there big spikes (holidays)?

###  Chart 2: Sales by day of week
Are Mondays slow? Are Saturdays the best? (Spoiler: Sundays are 0 because most stores are closed!)

 **Why visualize?** These charts often reveal **business insights** that the model alone can't communicate to your boss.

In [ ]:
df.groupby("Date")["Sales"].mean().plot(figsize=(12, 4), title="Average Daily Sales")
plt.show()

sns.boxplot(x="DayOfWeek", y="Sales", data=df)
plt.title("Sales by Day of Week"); plt.show()

## Step 7: Add "memory" features — lag and rolling averages

 **Big idea:** Past sales are the best predictor of future sales!

###  Lag features
*"What were sales 7, 14, or 30 days ago?"*

Why this matters:
- Sales **last week** → similar today (weekly patterns)
- Sales **2 weeks ago** → catches longer cycles
- Sales **30 days ago** → monthly patterns

###  Rolling averages
*"What was the average sales over the past 7 or 30 days?"*

This smooths out random noise. A single weird day shouldn't fool the model — but a 7-day average is reliable.

###  Why `groupby("Store")` and `shift(1)`?
- **`groupby("Store")`** — calculate features for each store separately (Store 1's history doesn't predict Store 2)
- **`shift(1)`** — only use info from BEFORE today, not today itself (would be cheating!)

###  `dropna()`
The first 30 rows of each store don't have 30 days of history yet, so their lag features are blank. We drop those rows.

In [ ]:
df = df.sort_values(["Store", "Date"]).reset_index(drop=True)

# Lag features (past sales per store)
df["Sales_lag_7"]  = df.groupby("Store")["Sales"].shift(7)
df["Sales_lag_14"] = df.groupby("Store")["Sales"].shift(14)
df["Sales_lag_30"] = df.groupby("Store")["Sales"].shift(30)

# Rolling averages (per store)
df["Sales_roll_7"]  = df.groupby("Store")["Sales"].shift(1).rolling(7).mean().reset_index(0, drop=True)
df["Sales_roll_30"] = df.groupby("Store")["Sales"].shift(1).rolling(30).mean().reset_index(0, drop=True)

# Drop rows that don't have enough history
df = df.dropna().reset_index(drop=True)

## Step 8: More feature engineering — variability & competition

We add a few more clever features:

###  Sales volatility (standard deviation)
`Sales_roll_7_std` — how *bumpy* were sales over the past 7 days? 
- **Low std** → consistent sales (predictable store)
- **High std** → wild swings (harder to predict)

###  Best recent sales
`Sales_roll_30_max` — what was the *peak* in the past 30 days? Helps the model gauge each store's potential.

###  How long has competition been around?
`CompetitionOpen` — turns "competitor opened in 2010-March" into "X months ago".
- Brand-new competition? Sales drop is fresh.
- Competition opened 5 years ago? Customers have already adjusted.

 **The pattern:** Good ML often comes from **smart feature engineering**, not fancy algorithms.

In [ ]:
# Sales trends per store
df["Sales_roll_7_std"] = df.groupby("Store")["Sales"].shift(1).rolling(7).std().reset_index(0, drop=True)
df["Sales_roll_30_max"] = df.groupby("Store")["Sales"].shift(1).rolling(30).max().reset_index(0, drop=True)

# Days since competition opened
df["CompetitionOpen"] = 12 * (df["Year"] - df["CompetitionOpenSinceYear"]) + \
                        (df["Month"] - df["CompetitionOpenSinceMonth"])
df["CompetitionOpen"] = df["CompetitionOpen"].apply(lambda x: x if x > 0 else 0)

# Clean up any new NaNs from rolling
df = df.dropna().reset_index(drop=True)

## Step 9: Time-based train/test split

###  Why NOT random split?

For most ML problems, we randomly split data into 80% train / 20% test. **For time-series, this is WRONG.** 

Why? Because **future depends on the past**. If we randomly split:
- The model might train on July data
- Then test on June data
- That's like asking "predict yesterday given tomorrow's info" → meaningless!

###  The right way: time-based split
We use the **last 6 weeks as the test set** and everything before that as training. This mimics real life:
- *"Train on what we know now"*
- *"Test if we can predict the next 6 weeks"*

###  Selected features
We use 21 features — combining store info, dates, promotions, and historical sales patterns.

In [ ]:
features = ["Store", "DayOfWeek", "Promo", "StateHoliday", "SchoolHoliday",
            "StoreType", "Assortment", "CompetitionDistance",
            "Year", "Month", "Day", "WeekOfYear", "IsWeekend",
            "Sales_lag_7", "Sales_lag_14", "Sales_lag_30",
            "Sales_roll_7", "Sales_roll_30",
            "Sales_roll_7_std", "Sales_roll_30_max", "CompetitionOpen"]
X = df[features]
y = df["Sales"]

# Use last 6 weeks as test set 
cutoff = df["Date"].max() - pd.Timedelta(weeks=6)
train_mask = df["Date"] <= cutoff

X_train, X_test = X[train_mask], X[~train_mask]
y_train, y_test = y[train_mask], y[~train_mask]

print("Train:", X_train.shape, "Test:", X_test.shape)

## Step 10: Train Model #1 — Linear Regression (baseline)

###  What is Linear Regression?
The simplest model. It tries to fit a **straight line** through the data:

> *Sales = (something) × Promo + (something) × DayOfWeek + ... + base*

It's like a **weighted scorecard** — each feature gets a weight, all weights add up to predict sales.

###  Why start with Linear Regression?
It's our **baseline** — the bar to beat. If a fancier model can't beat this simple one, we're doing something wrong.

###  Three metrics to evaluate:

| Metric | What it tells you | Better when |
|--------|-------------------|-------------|
| **MAE** (Mean Absolute Error) | On average, predictions are off by X dollars/sales | Lower  |
| **RMSE** (Root Mean Squared Error) | Like MAE but punishes big mistakes more | Lower  |
| **R²** | % of variation the model explains (0 to 1) | Higher  |

In [ ]:
lr = LinearRegression()
lr.fit(X_train, y_train)
lr_pred = lr.predict(X_test)

print("Linear Regression")
print(f"  MAE : {mean_absolute_error(y_test, lr_pred):.2f}")
print(f"  RMSE: {np.sqrt(mean_squared_error(y_test, lr_pred)):.2f}")
print(f"  R²  : {r2_score(y_test, lr_pred):.3f}")

## Step 11: Train Model #2 — XGBoost (the heavy hitter) 

###  What is XGBoost?
**XGBoost** = e**X**treme **G**radient **Boost**ing.

Imagine **2,000 simple decision trees**, each one correcting the mistakes of the previous one. Together, they form an extremely accurate predictor.

It's the **#1 most popular algorithm** for tabular data competitions on Kaggle. Many real-world ML systems use it.

###  Why log-transform Sales?
Sales values range from $20 to $40,000+ — a huge range. We use `np.log1p` (log + 1) to compress them:
- Makes errors comparable across small and large stores
- Prevents the model from obsessing over outliers
- Then `np.expm1` reverses it back to original scale at the end

###  The settings explained:

| Setting | What it means |
|---------|---------------|
| `n_estimators=2000` | Build up to 2,000 trees |
| `learning_rate=0.05` | Each tree adds 5% to the prediction (slow + steady = better) |
| `max_depth=8` | Each tree asks at most 8 questions before deciding |
| `subsample=0.9` | Each tree sees 90% random rows (prevents overfitting) |
| `colsample_bytree=0.9` | Each tree sees 90% random features |
| `early_stopping_rounds=50` | Stop early if no improvement after 50 trees |

 **Early stopping is smart** — it stops training as soon as performance plateaus, saving time.

In [ ]:
import numpy as np

y_train_log = np.log1p(y_train)
y_test_log = np.log1p(y_test)

xgb = XGBRegressor(
    n_estimators=2000,
    learning_rate=0.05,
    max_depth=8,
    subsample=0.9,
    colsample_bytree=0.9,
    random_state=42,
    n_jobs=-1,
    early_stopping_rounds=50
)

xgb.fit(X_train, y_train_log,
        eval_set=[(X_test, y_test_log)],
        verbose=False)

xgb_pred = np.expm1(xgb.predict(X_test))   # back to original scale

print(f"MAE : {mean_absolute_error(y_test, xgb_pred):.2f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, xgb_pred)):.2f}")
print(f"R²  : {r2_score(y_test, xgb_pred):.3f}")

## Step 12: Visualize predictions vs reality

Numbers tell us *how* good the model is. **Charts show us *where* it's good (and where it fails).**

We plot the **first 300 predictions** alongside actual sales:
-  **Blue line** — actual sales
-  **Orange line** — predicted sales

If the lines closely follow each other → the model captures the patterns well. If they diverge → there are patterns the model missed.

In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(y_test.values[:300], label="Actual")
plt.plot(xgb_pred[:300], label="Predicted")
plt.legend(); plt.title("Actual vs Predicted Sales (first 300)"); plt.show()

## Step 13: Find what drives sales — Feature Importance

 This is the **business gold** of the project!

Feature importance tells us: *"Which factors matter most in predicting sales?"*

It's like asking the model: *"Out of all the things you looked at, which ones helped you most?"*

###  Why this matters for the business

Knowing the top sales drivers helps the company **take action**:

-  If **lagged sales** are #1 → past patterns predict future (focus on consistency)
-  If **Promo** is critical → invest more in promotions
-  If **CompetitionDistance** matters → review locations near competitors
-  If **DayOfWeek** is huge → adjust staffing by day

The chart below shows features ranked from least to most important.

In [ ]:
imp = pd.Series(xgb.feature_importances_, index=features).sort_values()
imp.plot(kind="barh", figsize=(8, 5), title="Feature Importance")
plt.show()

## Step 14: Save the trained model

 We save our trained XGBoost model to a file so we can:
-  **Reuse it** without retraining (training takes minutes)
-  **Share it** with the marketing team
-  **Deploy it** in a real app or dashboard

###  What is a `.pkl` file?
It's a **serialized Python object** — basically the model's brain saved to disk.

Later, anyone can do:
```python
model = joblib.load("rossmann_xgb.pkl")
model.predict(...)   # instant predictions, no retraining needed!
```

In [ ]:
import joblib
joblib.dump(xgb, "rossmann_xgb.pkl")
print("Model saved as rossmann_xgb.pkl")

# Reload later with:
# model = joblib.load("rossmann_xgb.pkl")

## Step 15: Save outputs for the business team

We save **3 important CSV files** that other people (managers, analysts) can use without running the whole notebook:

### 1️ `sales_predictions.csv` — Daily predictions
For every day in the test set:
- **Date** — which day
- **Store** — which store
- **Actual_Sales** — what actually happened
- **Predicted_Sales** — what we predicted
- **Error** — the difference

 **Use case:** Operations team can identify which stores/days were hardest to predict.

### 2️ `model_metrics.csv` — Performance summary
Easy table with MAE, RMSE, R². Good for slide decks and reports.

### 3️ `feature_importance.csv` — Top sales drivers
Sorted list of which features mattered most. Useful for strategy.

In [ ]:
# 1. Save predictions (actual vs predicted)
results = pd.DataFrame({
    "Date": df.loc[~train_mask, "Date"].values,
    "Store": df.loc[~train_mask, "Store"].values,
    "Actual_Sales": y_test.values,
    "Predicted_Sales": xgb_pred.round(2),
    "Error": (y_test.values - xgb_pred).round(2)
})
results.to_csv("sales_predictions.csv", index=False)
print(" Predictions saved → sales_predictions.csv")

# 2. Save metrics summary
metrics = pd.DataFrame({
    "Metric": ["MAE", "RMSE", "R2"],
    "Value": [
        round(mean_absolute_error(y_test, xgb_pred), 2),
        round(np.sqrt(mean_squared_error(y_test, xgb_pred)), 2),
        round(r2_score(y_test, xgb_pred), 3)
    ]
})
metrics.to_csv("model_metrics.csv", index=False)
print(" Metrics saved → model_metrics.csv")

# 3. Save feature importance
importance = pd.DataFrame({
    "Feature": features,
    "Importance": xgb.feature_importances_
}).sort_values("Importance", ascending=False)
importance.to_csv("feature_importance.csv", index=False)
print(" Feature importance saved → feature_importance.csv")

##  Summary — what did we accomplish?

###  What we built
A machine learning model that predicts **daily sales for 1,115 stores** based on:
- Calendar features (year, month, day-of-week, holidays)
- Store characteristics (type, assortment, competition)
- **Past sales patterns** (most important!)
- Promotions

###  The 2 models we compared

| Model | Strength | Weakness |
|-------|----------|----------|
| Linear Regression | Simple, fast, interpretable | Misses non-linear patterns |
| **XGBoost**  | **Captures complex patterns, much higher accuracy** | Slower, less interpretable |

###  Key insights

1. **Time-based split is crucial** for time series — never randomly split dates!
2. **Lag features dominate** — past sales are the best predictor of future sales
3. **Feature engineering > algorithm tuning** — clever features beat fancy models
4. **Log transforms help** when target values span a huge range
5. **XGBoost crushes Linear Regression** for complex tabular data

###  Business recommendations

Based on what the model found:

-  **Use historical patterns** — past sales predict future, plan inventory accordingly
-  **Time promotions strategically** — Promo days have very different patterns
-  **Adjust staffing by day** — DayOfWeek strongly affects sales
-  **Watch competition** — distance and timing matter
-  **Run model weekly** — generate forecasts for inventory planning

###  Future improvements

-  **Hyperparameter tuning** with Optuna (target: lower RMSE)
-  **Try LightGBM and CatBoost** (often beat XGBoost)
-  **Add weather data** — rain affects retail!
-  **Holiday calendars** for each German state
-  **Per-store models** for top performers
-  **SHAP values** to explain individual predictions
-  **Deploy as web app** for real-time forecasting

---

##  Thanks for reading!

Hope this gave you a clear picture of how machine learning is used in **retail forecasting**. The same approach works for predicting:
-  Demand forecasting (Amazon, Walmart)
-  Ride-sharing demand (Uber, Lyft)
-  Hotel booking volumes
-  Streaming viewership
- ...and many more time-series problems!
